# Oracle-ROI 可行性 Pilot

**目的**：测一个数字 —— **"纯图像肾脏定位器" 在没见过的病例上，能盖住多少比例的肾脏**。

**为什么需要它**：现在的全部结果都是在"用真值标签定好裁剪框"的条件下得到的。
如果换成一个只能看 CT 的定位器，框会偏、会小、偶尔会完全找不到肾脏 —— 那么结果会掉多少？

这个 notebook 用 **约 1–2 小时**（大部分是下载时间）回答这个问题，而不是直接投入 2 周做完整修复。

---

## 怎么跑

1. 在 Jupyter Lab 里打开：`research/Oracle_ROI_Pilot.ipynb`
2. **从上到下依次运行**（Shift+Enter）
3. 第 3 格是**零成本分析**，不联网也能跑，30 秒出结果
4. 第 4 格开始需要联网下载数据

## 中断了怎么办

**直接重跑即可。** 下载和预处理都会跳过已完成的文件。

---

## 结果怎么判读

最后第 9 格会输出 `覆盖率`，对照下表：

| 覆盖率 | 含义 | 建议 |
|---|---|---|
| **≥ 98%** | 定位器几乎不出错 | 值得投入 2 周做完整修复（老师路线图的 P0） |
| **90–98%** | 会掉点，但可控 | 算一下预计掉多少，再决定 |
| **< 90%** | 定位器不可靠 | 立刻走"声明为 oracle-ROI 评估"的路线，省 2 周 |

## 背景：oracle-ROI 是什么

现在论文的流程是：

```
① 用真值标签算出肾脏包围盒   ← 部署时没有这个
② 按包围盒裁 192×192 的框
③ 模型只看 CT 像素 → 输出掩码   ← 这部分是真的
④ 算 Dice / 敏感性              ← 这部分是真的
```

第 ① 步用了只有标注时才有的信息，所以叫 **oracle-ROI**（假设完美 ROI 已知）。

审稿意见 1.1 要求：**改成纯图像定位**，或者**如实声明为 oracle-ROI 评估**。

这个 pilot 测的是：**如果补上第 ① 步，会掉多少分。**

In [1]:
# ============ 1. 配置 ============
import os, json, math, time, random
import numpy as np

ROOT       = r"E:\segmentation research paper"
CACHE      = os.path.join(ROOT, "_unzipped", "kits23_roi")
FOLD_JSON  = os.path.join(ROOT, "_unzipped", "runs", "folds_kits23_5_seed42.json")
RAW_DIR    = os.path.join(ROOT, "_unzipped", "kits23_raw")
PILOT_DIR  = os.path.join(ROOT, "_unzipped", "kits23_pilot")

FOLD      = 0            # 用哪一折做 pilot
N_TRAIN   = 120          # 下载多少例用于训练定位器
N_TEST    = 40           # 下载多少例用于测覆盖率
SHAPE     = (64, 128, 128)   # 定位器输入尺寸 (D, H, W)
EPOCHS    = 40
BATCH     = 2
BASE_CH   = 16
WORKERS   = 4            # 并行下载线程数（实测 4 线程约 2x 加速）

HF_REPO          = "MedOtter/kits23"
HF_REPO_FALLBACK = "Angelou0516/kits23"
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

for d in (RAW_DIR, PILOT_DIR):
    os.makedirs(d, exist_ok=True)

import torch, torch.nn as nn, torch.nn.functional as F
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", dev)
if dev.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

meta = json.load(open(FOLD_JSON, encoding="utf-8"))
CASES, FOLDS = meta["cases"], meta["folds"]
test_cases  = sorted(FOLDS[FOLD])
train_cases = sorted(set(CASES) - set(FOLDS[FOLD]))
print(f"\nfold {FOLD}: 训练池 {len(train_cases)} 例 | 测试 {len(test_cases)} 例")
print(f"本 pilot 使用: 训练 {min(N_TRAIN, len(train_cases))} 例, 测试 {min(N_TEST, len(test_cases))} 例")

torch 2.14.1+cu126 | cuda True | cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

fold 0: 训练池 391 例 | 测试 98 例
本 pilot 使用: 训练 120 例, 测试 40 例


## 第 3 格（零成本）：现有裁剪框里，肾脏离边界有多远

这一格**不需要下载任何数据**，直接读现有的 `kits23_roi` 缓存。

它回答一个前置问题：**现在的框有多宽松？** 框越宽松，定位器越容易命中。

> 注意：缓存里的标签包含**两侧肾脏**，所以要取"离框中心最近的那个连通域"作为目标肾脏，
> 否则量到的是对侧肾脏。

In [2]:
# ============ 3. 零成本分析：同侧肾脏到边界的余量 ============
from scipy import ndimage

def ipsilateral_margin(seg):
    """目标肾脏（离 ROI 中心最近的连通域）到 192x192 边界的最近距离（像素≈mm）。"""
    fg = seg > 0
    if not fg.any():
        return None
    cc, n = ndimage.label(fg)
    if n == 0:
        return None
    H, W = seg.shape[1], seg.shape[2]
    ctr = np.array([H / 2.0, W / 2.0])
    best, best_d = None, 1e9
    for i in range(1, n + 1):
        m = cc == i
        if m.sum() < 200:            # 忽略碎片
            continue
        ys, xs = np.nonzero(m.any(axis=0))
        c = np.array([ys.mean(), xs.mean()])
        d = float(np.linalg.norm(c - ctr))
        if d < best_d:
            best_d, best = d, m
    if best is None:
        return None
    ys, xs = np.nonzero(best.any(axis=0))
    m_y = min(ys.min(), H - 1 - ys.max())
    m_x = min(xs.min(), W - 1 - xs.max())
    return int(min(m_y, m_x)), int(m_y), int(m_x)

keys = sorted({f[:-8] for f in os.listdir(CACHE) if f.endswith("_seg.npy")})
margins = []
for k in keys:
    seg = np.load(os.path.join(CACHE, k + "_seg.npy"))
    r = ipsilateral_margin(seg)
    if r is not None:
        margins.append(r[0])

margins = np.array(margins)
print(f"  统计 {len(margins)} 个 ROI 体积")
print(f"  同侧肾脏到边界的最小余量（像素 ≈ mm）:")
print(f"    最小 {margins.min():3d} | 10% 分位 {int(np.percentile(margins,10)):3d} | "
      f"中位 {int(np.median(margins)):3d} | 均值 {margins.mean():.1f}")
print()
for thr in (0, 5, 10, 15, 20, 30):
    n = int((margins <= thr).sum())
    print(f"    余量 ≤ {thr:2d}mm 的体积: {n:4d} / {len(margins)}  ({100*n/len(margins):5.1f}%)")
print()
print("解读：余量越小，定位器的误差容忍度越低。")
print("      中位数较大 (>30mm) 说明框普遍宽松，定位器容易成功。")

  统计 968 个 ROI 体积
  同侧肾脏到边界的最小余量（像素 ≈ mm）:
    最小   0 | 10% 分位  41 | 中位  55 | 均值 52.4

    余量 ≤  0mm 的体积:    6 / 968  (  0.6%)
    余量 ≤  5mm 的体积:    6 / 968  (  0.6%)
    余量 ≤ 10mm 的体积:    6 / 968  (  0.6%)
    余量 ≤ 15mm 的体积:   10 / 968  (  1.0%)
    余量 ≤ 20mm 的体积:   18 / 968  (  1.9%)
    余量 ≤ 30mm 的体积:   44 / 968  (  4.5%)

解读：余量越小，定位器的误差容忍度越低。
      中位数较大 (>30mm) 说明框普遍宽松，定位器容易成功。


## 第 4 格：检查网络并下载数据

- 需要能访问 `hf-mirror.com`（国内镜像）
- 只下载 pilot 需要的一小部分：约 160 例 ≈ 4–5 GB
- **已有文件会跳过**，中断后重跑无损失

如果这一格失败（网络问题），**第 3 格的结果仍然有效**，可以先把它发给老师。

In [ ]:
# ============ 4. 下载（并行，可续） ============
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor, as_completed

need_train = train_cases[:N_TRAIN]
need_test  = test_cases[:N_TEST]
need = list(need_train) + list(need_test)

def have(cid):
    return (os.path.exists(os.path.join(RAW_DIR, "dataset", cid, "imaging.nii.gz"))
            and os.path.exists(os.path.join(RAW_DIR, "dataset", cid, "segmentation.nii.gz")))

todo = [c for c in need if not have(c)]
print(f"需要下载 {len(todo)} 例（已有 {len(need)-len(todo)} 例）")
print(f"并行线程数: {WORKERS}")

def fetch(cid):
    last = None
    for r in (HF_REPO, HF_REPO_FALLBACK):
        try:
            for fn in ("imaging.nii.gz", "segmentation.nii.gz"):
                hf_hub_download(r, f"dataset/{cid}/{fn}", repo_type="dataset", local_dir=RAW_DIR)
            return (cid, None)
        except Exception as e:
            last = str(e)[:80]
    return (cid, last)

t0 = time.time()
done_n, failed = 0, []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(fetch, c) for c in todo]
    for f in as_completed(futs):
        cid, err = f.result()
        done_n += 1
        if err:
            failed.append((cid, err))
        if done_n % 10 == 0 or done_n == len(todo):
            el = time.time() - t0
            rate = el / max(done_n, 1)
            eta = rate * (len(todo) - done_n) / 60
            print(f"  {done_n}/{len(todo)}  {el/60:5.1f} min  "
                  f"{rate:4.1f}s/例  剩余约 {eta:4.0f} min  失败 {len(failed)}", flush=True)

print(f"\n下载完成，用时 {(time.time()-t0)/60:.1f} min，失败 {len(failed)} 例")
if failed:
    print("  失败示例:", failed[:3])

C:\conda_envs\segkid\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


需要下载 141 例（已有 19 例）
并行线程数: 4


HTTP Error 429 thrown while requesting GET https://hf-mirror.com/api/datasets/MedOtter/kits23/xet-read-token/ba6ad1e37f5dae6e555d3721a02b10496fa8431f
Rate limited. Waiting 34.0s before retry [Retry 1/5].


  10/141    4.5 min  26.9s/例  剩余约   59 min  失败 0


Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: The read operation timed out)


  20/141    9.5 min  28.4s/例  剩余约   57 min  失败 0
  30/141   18.5 min  37.1s/例  剩余约   69 min  失败 0


## 第 5 格：预处理成固定尺寸

把原始 CT 重采样到 `64 × 128 × 128`（全腹部视野），标签用最近邻。

- 图像：HU 窗 [-200, 300] 归一化（肾脏是软组织，这个窗最清楚）
- 标签：`lab > 0`（肾脏，含肿瘤和囊肿）

第一次跑要对 160 例做重采样，约 **3–5 分钟**；之后走缓存。

In [ ]:
# ============ 5. 预处理 ============
import nibabel as nib

def _fit(a, shape, order):
    z = [shape[i] / a.shape[i] for i in range(3)]
    b = ndimage.zoom(a.astype(np.float32), z, order=order)
    out = np.zeros(shape, np.float32)
    sl = tuple(slice(0, min(shape[i], b.shape[i])) for i in range(3))
    out[sl] = b[sl]
    return out

def prep(cid):
    dst = os.path.join(PILOT_DIR, cid + ".npz")
    if os.path.exists(dst):
        d = np.load(dst)
        return d["x"], d["y"]
    ip = os.path.join(RAW_DIR, "dataset", cid, "imaging.nii.gz")
    sp = os.path.join(RAW_DIR, "dataset", cid, "segmentation.nii.gz")
    hu  = np.asarray(nib.load(ip).dataobj, dtype=np.float32)
    lab = np.asarray(nib.load(sp).dataobj).astype(np.float32)
    hu  = _fit(hu, SHAPE, 1)
    lab = _fit(lab, SHAPE, 0)
    hu  = np.clip(hu, -200.0, 300.0) / 250.0 - 0.4
    y   = (lab > 0.5).astype(np.float32)
    np.savez_compressed(dst, x=hu.astype(np.float32), y=y.astype(np.float32))
    return hu.astype(np.float32), y

t0 = time.time()
Xtr, Ytr, Xte, Yte, te_ids = [], [], [], [], []
for c in need_train:
    try:
        x, y = prep(c); Xtr.append(x); Ytr.append(y)
    except Exception as e:
        print("  skip train", c, str(e)[:60])
for c in need_test:
    try:
        x, y = prep(c); Xte.append(x); Yte.append(y); te_ids.append(c)
    except Exception as e:
        print("  skip test", c, str(e)[:60])

Xtr = np.stack(Xtr)[:, None]; Ytr = np.stack(Ytr)[:, None]
Xte = np.stack(Xte)[:, None]; Yte = np.stack(Yte)[:, None]
print(f"训练 {Xtr.shape}  测试 {Xte.shape}  用时 {(time.time()-t0):.1f}s")

## 第 6–7 格：训练一个低分辨率 3D U-Net 定位器

这个网络**只做一件事**：粗定位肾脏。它不需要精确，只需要给出一个足够大的框。

- 结构：3 层下采样的小 U-Net（约 1–2 M 参数）
- 损失：BCE + soft Dice
- 40 个 epoch，预计 **10–15 分钟**

In [ ]:
# ============ 6. 模型 ============
def cbr3(i, o):
    return nn.Sequential(nn.Conv3d(i, o, 3, padding=1, bias=False),
                         nn.InstanceNorm3d(o, affine=True),
                         nn.LeakyReLU(0.01, True))

class LocalizerUNet(nn.Module):
    def __init__(self, base=BASE_CH):
        super().__init__()
        self.e1 = nn.Sequential(cbr3(1, base), cbr3(base, base))
        self.e2 = nn.Sequential(cbr3(base, base*2), cbr3(base*2, base*2))
        self.e3 = nn.Sequential(cbr3(base*2, base*4), cbr3(base*4, base*4))
        self.b  = nn.Sequential(cbr3(base*4, base*8), cbr3(base*8, base*8))
        self.d3 = nn.Sequential(cbr3(base*8 + base*4, base*4), cbr3(base*4, base*4))
        self.d2 = nn.Sequential(cbr3(base*4 + base*2, base*2), cbr3(base*2, base*2))
        self.d1 = nn.Sequential(cbr3(base*2 + base, base), cbr3(base, base))
        self.out = nn.Conv3d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool3d(e1, 2))
        e3 = self.e3(F.max_pool3d(e2, 2))
        b  = self.b(F.max_pool3d(e3, 2))
        u3 = self.d3(torch.cat([F.interpolate(b,  size=e3.shape[2:], mode="trilinear", align_corners=False), e3], 1))
        u2 = self.d2(torch.cat([F.interpolate(u3, size=e2.shape[2:], mode="trilinear", align_corners=False), e2], 1))
        u1 = self.d1(torch.cat([F.interpolate(u2, size=e1.shape[2:], mode="trilinear", align_corners=False), e1], 1))
        return self.out(u1)

net = LocalizerUNet().to(dev)
print("params: %.2f M" % (sum(p.numel() for p in net.parameters()) / 1e6))

In [ ]:
# ============ 7. 训练 ============
def dice_loss(p, y, eps=1.0):
    p = p.sigmoid()
    num = 2 * (p * y).sum() + eps
    den = p.sum() + y.sum() + eps
    return 1 - num / den

opt = torch.optim.Adam(net.parameters(), lr=1e-3)
n = len(Xtr)
print(f"training on {n} volumes, {EPOCHS} epochs")
t0 = time.time()
for ep in range(EPOCHS):
    net.train()
    perm = np.random.permutation(n)
    tot, tot_d = 0.0, 0.0
    for i in range(0, n, BATCH):
        idx = perm[i:i+BATCH]
        x = torch.from_numpy(Xtr[idx]).to(dev)
        y = torch.from_numpy(Ytr[idx]).to(dev)
        logit = net(x)
        loss = F.binary_cross_entropy_with_logits(logit, y) + dice_loss(logit, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        tot += float(loss); tot_d += float(dice_loss(logit, y))
    if (ep + 1) % 5 == 0 or ep == 0:
        print(f"  ep{ep+1:02d}  loss={tot/(n/BATCH):.4f}  dice_loss={tot_d/(n/BATCH):.4f}  "
              f"{(time.time()-t0)/60:.1f} min")
print(f"训练完成，用时 {(time.time()-t0)/60:.1f} min")

## 第 8 格：覆盖率评估（核心）

对每个**没见过的**测试病例：

1. 定位器预测肾脏概率图 → 阈值 0.5 → 取最大连通域
2. 用这个连通域的包围盒作为"定位器给出的框"
3. 计算：**真值肾脏有多少体积落在框内** = `覆盖率`

同时报告：
- **检出率**：定位器是否找到了肾脏（预测非空）
- **覆盖率均值 / 中位数**
- **覆盖率 ≥95% 的比例**

最后给出 **go / no-go**。

In [ ]:
# ============ 8. 覆盖率评估 ============
@torch.no_grad()
def predict_mask(x):
    net.eval()
    logit = net(torch.from_numpy(x[None]).to(dev))   # x is (1, D, H, W) -> add batch dim only
    return (logit[0, 0].sigmoid().cpu().numpy() > 0.5)

def bbox_of(m):
    idx = np.argwhere(m)
    if len(idx) == 0:
        return None
    return idx.min(0), idx.max(0)

rows = []
for j, cid in enumerate(te_ids):
    y = Yte[j, 0].astype(bool)
    if y.sum() == 0:
        continue                                    # 真值无肾脏（异常），跳过
    pm = predict_mask(Xte[j])
    bb = bbox_of(pm)
    if bb is None:
        rows.append(dict(case=cid, detected=False, coverage=0.0))
        continue
    lo, hi = bb
    keep = np.zeros_like(y)
    keep[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1] = True
    cov = float((y & keep).sum()) / float(y.sum())
    rows.append(dict(case=cid, detected=True, coverage=cov))

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv(os.path.join(PILOT_DIR, "pilot_coverage.csv"), index=False)

det = df.detected.mean()
cov = df.coverage.values
print(f"  测试病例: {len(df)}")
print(f"  检出率 (定位器找到肾脏): {100*det:.1f}%")
print(f"  覆盖率 均值: {100*cov.mean():.1f}%   中位: {100*np.median(cov):.1f}%   最小: {100*cov.min():.1f}%")
print()
for thr in (0.99, 0.95, 0.90, 0.80, 0.50):
    print(f"    覆盖率 ≥ {int(thr*100):2d}% 的病例: {int((cov>=thr).sum()):3d} / {len(cov)}  "
          f"({100*(cov>=thr).mean():5.1f}%)")
print()
mean_cov = 100 * float(cov.mean())
if mean_cov >= 98:
    verdict, advice = "GO", "定位器几乎不出错 → 值得投入做完整修复（老师路线图的 P0）"
elif mean_cov >= 90:
    verdict, advice = "CONDITIONAL", "会掉点但可控 → 估算掉多少再决定"
else:
    verdict, advice = "NO-GO", "定位器不可靠 → 建议走'声明为 oracle-ROI 评估'的路线"
print("=" * 70)
print(f"  结论: {verdict}   平均覆盖率 {mean_cov:.1f}%")
print(f"  建议: {advice}")
print("=" * 70)
print(f"\n逐病例结果已保存: {os.path.join(PILOT_DIR, 'pilot_coverage.csv')}")

## 第 9 格：把结果整理成一段话（可发给老师 / 写进论文）

跑完上面的格子后运行这一格。

In [ ]:
# ============ 9. 生成结论文字 ============
try:
    mean_cov, det
except NameError:
    import pandas as pd
    df = pd.read_csv(os.path.join(PILOT_DIR, "pilot_coverage.csv"))
    mean_cov, det = 100*df.coverage.mean(), df.detected.mean()

n_tr = len(Xtr)
text = f'''An image-only 3-D kidney localizer was trained on {n_tr} training cases of
fold {FOLD} and evaluated on {len(te_ids)} held-out cases. The localizer detected a kidney in
{100*det:.1f}% of cases, and the bounding box it produced contained a mean of {mean_cov:.1f}% of the
true kidney volume (median {100*float(np.median(cov)):.1f}%, minimum {100*float(cov.min()):.1f}%).
{int((cov>=0.95).sum())} of {len(cov)} cases had at least 95% of the kidney inside the predicted box.'''
print(text)
print()
print("可直接粘贴的英文（按实际数字调整）:")
print("-" * 70)
print(text)